# Dataset Augmentation for Rare Allergen Classes

**Purpose:** Targeted oversampling of rare allergen classes using synonym replacement to improve class balance for model training.

Expands tree_nuts (1.5%
ightarrow~7%), shellfish (3%
ightarrow~9%), fish (7%
ightarrow~11%), eggs (9%
ightarrow~12%), and peanuts (8%
ightarrow~11%).

In [1]:
import os
import sys
import json
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Project utilities
from utils.data_utils import get_data_directories, augment_targeted
from utils.text_processing import get_allergen_list, allergens_to_binary, clean_html
from utils.evaluation import compute_per_class_metrics

SEED = 42
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 1. Load Dataset

In [2]:
dirs = get_data_directories()
input_path = os.path.join(dirs["final"], "labeled_dataset_enhanced.csv")
df = pd.read_csv(input_path)
print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
df.head(2)

Loaded dataset: 1057 rows, 11 columns
Columns: ['code', 'brands', 'product_name_en', 'ingredients_text_en', 'detected_allergens', 'official_allergens_mapped', 'traces_allergens', 'consensus_allergens', 'combined_allergens', 'may_contain', 'coconut']


,code,brands,product_name_en,ingredients_text_en,detected_allergens,official_allergens_mapped,traces_allergens,consensus_allergens,combined_allergens,may_contain,coconut
0,4806502720615,Gardenia,Gardenia White Bread Classic 600G,"wheat flour (vitamin b2, vitamin b3, vitamin a...","['milk', 'wheat']","['milk', 'wheat']",['soy'],"['milk', 'wheat']","{'detected_only': [], 'detected_or_official': ...",[],['coconut']
1,2116136,Signature Select,Slightly Dipped Almonds: French Vanilla Dark C...,"almonds, semi-sweet chocolate (unsweetened cho...",['milk'],"['soy', 'tree_nuts']",[],[],"{'detected_only': ['milk'], 'detected_or_offic...",[],[]


## 2. Create Binary Labels from Detected Allergens

In [3]:
# Clean ingredient text
df["ingredients_cleaned"] = df["ingredients_text_en"].apply(clean_html)

# Convert detected allergens to binary label vectors
allergens = get_allergen_list()
df["labels"] = df["detected_allergens"].apply(allergens_to_binary)

# Validate
label_array = np.array(df["labels"].tolist())
print(f"Label matrix shape: {label_array.shape}")
print(f"Per-class counts: {label_array.sum(axis=0).astype(int)}")
for i, a in enumerate(allergens):
    cnt = int(label_array[:, i].sum())
    pct = cnt / len(df) * 100
    print(f"  {a:12s}: {cnt:4d} positive ({pct:.1f}%)")

Label matrix shape: (1057, 8)
Per-class counts: [495 102  89  15 277 358  77  33]
  milk        :  495 positive (46.8%)
  eggs        :  102 positive (9.6%)
  peanuts     :   89 positive (8.4%)
  tree_nuts   :   15 positive (1.4%)
  soy         :  277 positive (26.2%)
  wheat       :  358 positive (33.9%)
  fish        :   77 positive (7.3%)
  shellfish   :   33 positive (3.1%)


## 3. Class Balance Before Augmentation

In [4]:
# Class balance summary
allergen_list = get_allergen_list()
label_array = np.array(df["labels"].tolist())
n = len(df)

print("=" * 55)
print(f"{'Allergen':12s} {'Count':>6s} {'%':>7s} {'Status':>10s}")
print("=" * 55)
for i, a in enumerate(allergen_list):
    cnt = int(label_array[:, i].sum())
    pct = cnt / n * 100
    status = "OK" if pct >= 10 else ("LOW" if pct >= 5 else "CRITICAL")
    print(f"{a:12s} {cnt:6d} {pct:6.1f}% {status:>10s}")
print("=" * 55)
print(f"Total samples: {n}")

Allergen      Count       %     Status
milk            495   46.8%         OK
eggs            102    9.6%        LOW
peanuts          89    8.4%        LOW
tree_nuts        15    1.4%   CRITICAL
soy             277   26.2%         OK
wheat           358   33.9%         OK
fish             77    7.3%        LOW
shellfish        33    3.1%   CRITICAL
Total samples: 1057


## 4. Targeted Augmentation

Applying per-class multipliers to oversample rare allergens using synonym replacement:
- tree_nuts (1.5%): 6x → ~9%
- shellfish (3%): 4x → ~12%
- fish (7%): 2x → ~18%
- eggs (9%): 1.5x → ~20%
- peanuts (8%): 1.5x → ~17%

In [5]:
# Define target allergens and their multipliers
target_allergens = {
    "tree_nuts": 6,
    "shellfish": 4,
    "fish": 2,
    "eggs": 1.5,
    "peanuts": 1.5
}

# Ensure text column exists for augment_targeted
# (it looks for 'text' then 'ingredients_cleaned')
df["text"] = df["ingredients_cleaned"]

# Run targeted augmentation
df_augmented = augment_targeted(
    df,
    target_allergens=target_allergens,
    ALLERGENS=allergen_list,
    random_state=SEED
)

print(f"Original samples: {len(df)}")
print(f"After augmentation: {len(df_augmented)}")
print(f"Added: {len(df_augmented) - len(df)} synthetic samples")

Original samples: 1057
After augmentation: 1624
Added: 567 synthetic samples


## 5. Class Balance After Augmentation

In [6]:
aug_label_array = np.array(df_augmented["labels"].tolist())
n_aug = len(df_augmented)
print("=" * 55)
print(f"{'Allergen':12s} {'Count':>6s} {'%':>7s} {'Change':>10s}")
print("=" * 55)
for i, a in enumerate(allergen_list):
    cnt_before = int(label_array[:, i].sum())
    cnt_after = int(aug_label_array[:, i].sum())
    pct_before = cnt_before / len(df) * 100
    pct_after = cnt_after / n_aug * 100
    change = pct_after - pct_before
    print(f"{a:12s} {cnt_after:6d} {pct_after:6.1f}% {change:+7.1f}%")
print("=" * 55)
print(f"Total samples: {n_aug}")

Allergen      Count       %     Change
milk            833   51.3%    +4.5%
eggs            355   21.9%   +12.2%
peanuts         293   18.0%    +9.6%
tree_nuts       121    7.5%    +6.0%
soy             606   37.3%   +11.1%
wheat           723   44.5%   +10.7%
fish            339   20.9%   +13.6%
shellfish       216   13.3%   +10.2%
Total samples: 1624


## 6. Save Augmented Dataset

In [7]:
# Prepare output columns
output_cols = [c for c in df_augmented.columns if c not in ["text", "labels"]]
output_df = df_augmented[output_cols].copy()

# Save
output_path = os.path.join(dirs["final"], "labeled_dataset_augmented.csv")
output_df.to_csv(output_path, index=False)

# Also save a version with label vectors for easy loading in training notebooks
labels_output_path = os.path.join(dirs["final"], "labeled_dataset_augmented_with_labels.csv")
save_df = df_augmented[["text", "labels"] + output_cols].copy()
# labels may already be python lists, not numpy arrays
save_df["labels"] = save_df["labels"].apply(lambda x: list(x) if hasattr(x, 'tolist') else x)
save_df.to_csv(labels_output_path, index=False)

print(f"Saved augmented dataset to {output_path}")
print(f"Saved with labels to {labels_output_path}")
print(f"Done. Dataset ready for Notebook 07 (allergen training) re-run.")

Saved augmented dataset to /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/data/final/labeled_dataset_augmented.csv
Saved with labels to /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/data/final/labeled_dataset_augmented_with_labels.csv
Done. Dataset ready for Notebook 07 (allergen training) re-run.


## 7. Summary Report

In [8]:
print("=" * 60)
print("Phase 3 — Dataset Augmentation Complete")
print("=" * 60)
print(f"Original size: {len(df)}")
print(f"Augmented size: {len(df_augmented)}")
print(f"Increase: {(len(df_augmented) / len(df) - 1) * 100:.0f}%")
print()
print("Improvements:")
for a in ["tree_nuts", "shellfish", "fish", "eggs", "peanuts"]:
    i = allergen_list.index(a)
    before = int(label_array[:, i].sum())
    after = int(aug_label_array[:, i].sum())
    print(f"  {a:12s}: {before:3d} → {after:3d} positive samples")
print()
print("Next steps: retrain model using labeled_dataset_augmented.csv in Notebook 07")

Phase 3 — Dataset Augmentation Complete
Original size: 1057
Augmented size: 1624
Increase: 54%

Improvements:
  tree_nuts   :  15 → 121 positive samples
  shellfish   :  33 → 216 positive samples
  fish        :  77 → 339 positive samples
  eggs        : 102 → 355 positive samples
  peanuts     :  89 → 293 positive samples

Next steps: retrain model using labeled_dataset_augmented.csv in Notebook 07
